# Rerank RAG - "Os Sertões" de Euclides da Cunha

Implementação de RAG com **re-ranking**, que adiciona uma etapa de refino após a busca inicial:

1. **Retrieval inicial**: busca ampla (top-20) por similaridade cosseno
2. **Re-ranking**: um modelo cross-encoder (FlashRank) re-ordena os resultados pela relevância real com a query
3. **Top-k final**: apenas os melhores chunks re-rankeados vão para a LLM

A vantagem: embeddings bi-encoder são rápidos mas aproximados; cross-encoders são mais precisos porque analisam query+documento juntos.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "Configure OPENAI_API_KEY no .env ou como variável de ambiente"

## 1. Carregamento e Chunking

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_PATH = "os-sertoes.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(pages)

print(f"Páginas: {len(pages)}")
print(f"Chunks: {len(chunks)}")

## 2. Embeddings e Vector Store

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

CHROMA_DIR = "./chroma_rerank"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name="os_sertoes_rerank"
)

print(f"Vector store criado com {vectorstore._collection.count()} documentos")

## 3. Configuração do Re-ranker (FlashRank)

FlashRank é um cross-encoder leve e rápido. Funciona offline e suporta português via modelos multilingual.

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import FlashrankRerank

base_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 20}
)

compressor = FlashrankRerank(
    top_n=6,
    model="ms-marco-MultiBERT-L-12"
)

rerank_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

print("Rerank retriever configurado!")
print(f"  - Busca inicial: top-20 por similaridade")
print(f"  - Re-ranking: FlashRank (ms-marco-MultiBERT-L-12)")
print(f"  - Resultado final: top-6 re-rankeados")

## 4. Comparação: Antes vs Depois do Re-ranking

Vamos visualizar como o re-ranking reordena os resultados.

In [ ]:
query_teste = "Antônio Conselheiro e seu papel na Guerra de Canudos"

print("=== ANTES do Re-ranking (top-6 por similaridade bruta) ===")
base_results = base_retriever.invoke(query_teste)
for i, doc in enumerate(base_results[:6], 1):
    print(f"\n{i}. [Pág. {doc.metadata.get('page', '?')}] {doc.page_content[:150]}...")

print(f"\n{'='*80}")

print("\n=== DEPOIS do Re-ranking (top-6 por cross-encoder) ===")
rerank_results = rerank_retriever.invoke(query_teste)
for i, doc in enumerate(rerank_results, 1):
    score = doc.metadata.get("relevance_score", "N/A")
    print(f"\n{i}. [Pág. {doc.metadata.get('page', '?')} | Score: {score:.4f}] {doc.page_content[:150]}...")

## 5. Configuração da Chain RAG com Rerank

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
    max_tokens=2000
)

PROMPT_TEMPLATE = """
Você é um especialista em literatura brasileira, especificamente na obra "Os Sertões" de Euclides da Cunha.
Responda a pergunta com base EXCLUSIVAMENTE nos trechos do livro fornecidos abaixo.
Os trechos foram selecionados e ordenados por relevância usando um modelo de re-ranking.
Se a informação não estiver nos trechos, diga que não encontrou a informação no contexto fornecido.
Cite trechos relevantes quando possível.

Contexto (ordenado por relevância):
{context}

Pergunta: {question}

Resposta detalhada:
"""

prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[Trecho {i+1} — Pág. {d.metadata.get('page', '?')} | Relevância: {d.metadata.get('relevance_score', 'N/A'):.4f}]\n{d.page_content}"
        for i, d in enumerate(docs)
    )

rerank_rag_chain = (
    {"context": rerank_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Pipeline Rerank RAG configurado com sucesso!")

## 6. Execução das Perguntas

In [ ]:
PERGUNTAS = [
    "Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?",
    "Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?",
    "Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?",
    "Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?",
    "Quais são os principais aspectos da crítica social e política presentes em 'Os Sertões'? Como esses aspectos refletem a visão do autor sobre o Brasil da época?"
]

In [ ]:
respostas_rerank = {}

for i, pergunta in enumerate(PERGUNTAS, 1):
    print(f"{'='*80}")
    print(f"PERGUNTA {i}: {pergunta}")
    print(f"{'='*80}")
    
    resposta = rerank_rag_chain.invoke(pergunta)
    respostas_rerank[i] = resposta
    
    print(f"\nRESPOSTA:\n{resposta}")
    print()

## 7. Análise dos Scores de Re-ranking

In [ ]:
for i, pergunta in enumerate(PERGUNTAS, 1):
    print(f"\n{'='*80}")
    print(f"PERGUNTA {i}: {pergunta[:80]}...")
    print(f"{'='*80}")
    
    docs = rerank_retriever.invoke(pergunta)
    
    for j, doc in enumerate(docs, 1):
        score = doc.metadata.get("relevance_score", 0)
        bar = "\u2588" * int(score * 50)
        print(f"  {j}. Score: {score:.4f} |{bar}| Pág. {doc.metadata.get('page', '?')} | {doc.page_content[:80]}...")